# Validation Visuelle du Slicer PixelOdyssey

In [ ]:
import os
import sys
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Configuration des chemins système pour importer notre code modulaire
# On remonte d'un cran pour atteindre la racine du projet
sys.path.append(os.path.abspath(os.path.join("..")))
from src.data.slicer import PlasticImageSlicer

ImportError: cannot import name 'PlasticImageSlicer' from 'src.data.slicer' (c:\Users\alexa\Documents\Jame\PixelOdyssey\src\data\slicer.py)

In [ ]:
# 2. Définition des dossiers sur ton SSD Samsung (D:)
RAW_DIR = r"D:\PixelOdyssey_Data\plastic_dataset\raw"
TEST_IMG_DIR = r"D:\PixelOdyssey_Data\plastic_dataset\images\test_tiles"
TEST_LAB_DIR = r"D:\PixelOdyssey_Data\plastic_dataset\labels\test_tiles"

os.makedirs(TEST_IMG_DIR, exist_ok=True)
os.makedirs(TEST_LAB_DIR, exist_ok=True)


In [ ]:
# 3. Initialisation du slicer avec tes paramètres de précision
slicer = PlasticImageSlicer(
    tile_size=640, 
    overlap=256, 
    discard_truncated=True  # Notre nouvelle stratégie anti-arêtes droites
)

# Nom de ton image test présente dans ton dossier RAW
# !!! Remplace par le nom d'un fichier réel présent dans ton dossier raw !!!
test_filename = "transect_A" 

img_path = os.path.join(RAW_DIR, f"{test_filename}.png")
label_path = os.path.join(RAW_DIR, f"{test_filename}.txt")

# Exécution du découpage
slicer.slice_single_pair(
    img_path=img_path,
    label_path=label_path,
    output_img_dir=TEST_IMG_DIR,
    output_label_dir=TEST_LAB_DIR,
    prefix=test_filename
)


In [ ]:
# 4. Visualisation d'une tuile générée pour valider le repositionnement des polygones
# Nous allons chercher la première tuile qui contient une annotation
annotated_tiles = [f for f in os.listdir(TEST_LAB_DIR) if f.endswith(".txt") and os.path.getsize(os.path.join(TEST_LAB_DIR, f)) > 0]

if not annotated_tiles:
    print("[WARN] Aucune tuile avec annotation trouvée. Vérifie que ton image test contient bien des déchets labellisés.")
else:
    # On prend la première tuile annotée pour la démonstration
    target_label = annotated_tiles[0]
    target_img = target_label.replace(".txt", ".png")
    
    tile_img_path = os.path.join(TEST_IMG_DIR, target_img)
    tile_lab_path = os.path.join(TEST_LAB_DIR, target_label)
    
    # Lecture de la tuile (OpenCV BGR -> RGB pour Matplotlib)
    image = cv2.imread(tile_img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)
    
    # Lecture et superposition du polygone YOLO
    with open(tile_lab_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            class_id = parts[0]
            coords = [float(x) for x in parts[1:]]
            
            # Reconstruction des points (x, y) absolus dans la tuile 640x640
            points = []
            for i in range(0, len(coords), 2):
                x_pixel = coords[i] * w
                y_pixel = coords[i+1] * h
                points.append([x_pixel, y_pixel])
            
            # Dessin du polygone
            polygon_patch = patches.Polygon(
                points, 
                closed=True, 
                linewidth=2, 
                edgecolor="cyan", 
                facecolor="cyan", 
                alpha=0.3, 
                label=f"Classe {class_id}"
            )
            ax.add_patch(polygon_patch)
            
    plt.title(f"Validation Visuelle : {target_img}")
    plt.legend()
    plt.axis("off")
    plt.show()